<a href="https://colab.research.google.com/github/Ateeqaslam/F--RAG-Pipeline/blob/main/RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install chromadb
!pip install pypdf
!pip install langchain-text-splitters
!pip install tenacity

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 741.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found

In [4]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tenacity import retry, stop_after_attempt, wait_exponential
import os, chromadb, re
import chromadb.utils.embedding_functions as chromadb_Embedding_Function
from google.colab import userdata
from google import genai
from google.genai import types
# API Key
os.environ["GEMINI_API_KEY"] = userdata.get('RAGPipeline')


DB_client = chromadb.Client()
Gemini = genai.Client()

system_prompt = (
        "You are an expert document reading assistant. Answer the user's question using ONLY the provided Context fragments. If the answer cannot be confidently derived from the context, say: {I cannot find that information in the provided document.}"
   )

User_prompt = "Does this document has my email? test@test.com and this is my phone number 03000000000 and this is my card number 4000 0000 0000 0000"

User_PDF = "Test.pdf"

# PDF Extraction
def extract_text_from_pdf(path):
    reader = PdfReader(path)

    Full_text = ""

    for page in reader.pages:
        text = page.extract_text()
        if text:
            Full_text += text + "\n"

    return Full_text

# Input Validation
def input_val(prompt: str):
    blocked_keywords = ["execute code", "bypass prompt",
                    "ignore previous instructions", "bypass system prompts",
                    "Sexual", "CSAM", "Hate", "Harassment", "Dangerous",
                    "Toxic", "Violent", "Profanity", "Illicit",
                    "hack", "exploit", "malware", "phishing", "bypass", "scrape", "circumvent", "rootkit", "ddos",
                    "suicide", "bomb", "kill", "assassinate", "torture", "mutilate", "weapons", "firearms",
                    "illicit drugs", "meth", "cocaine", "unlicensed pharmacy", "counterfeit bills", "dox", "scrape personal info", "biometric tracking", "spyware"]

    if any(words.lower() in prompt.lower() for words in blocked_keywords):
        raise ValueError("Security Violation: Unauthorized input sequence detected.")

# PII Redaction
def redact_sen_data(text: str):
    # regex
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    phone_pattern = r'(\+92|0092|92|0)?3[0-9]{9}'
    card_pattern = r'\b(?:\d[ -]*?){13,16}\b'

    text = re.sub(email_pattern, '[Redacted Email]', text)
    text = re.sub(phone_pattern, '[Redacted Phone]', text)
    text = re.sub(card_pattern, '[Redacted Card]', text)

    return text

# Text Splits using LangChain's RTS
def text_splitter(text: str):
    splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 20
    )
    chunks = splitter.split_text(text)

    return chunks

# Model call with API Error Handling
@retry(stop=stop_after_attempt(5), wait=wait_exponential(multiplier=2, min=2, max=10))
def call_model(prompt):
    response = Gemini.models.generate_content(
    model= "gemini-3.6-flash",
    contents= prompt,
    config= types.GenerateContentConfig(
        system_instruction= system_prompt,
        temperature=0.1
    ))
    return response


text = extract_text_from_pdf(User_PDF) # text from PDF

redacted_text = redact_sen_data(text) # redact document texts

chunks =  text_splitter(redacted_text) # chunks
chunk_ids = [f'chunk_{i}' for i in range(len(chunks))] # chunk's id

collection_doc = DB_client.get_or_create_collection(
    name = 'User_doc',
    embedding_function = chromadb_Embedding_Function.GoogleGenaiEmbeddingFunction(
        model_name= "gemini-embedding-001",
        task_type= "RETRIEVAL_DOCUMENT"
    )
)
collection_doc.add(
    documents = chunks,
    ids =  chunk_ids
)

# Input Guard
input_val(User_prompt)
# Redact PII
redacted_prompt = redact_sen_data(User_prompt)
collection_query_results = collection_doc.query(
    query_texts = redacted_prompt,
    n_results = 10
)


context = "\n".join([f"- {doc_results}" for doc_results in collection_query_results["documents"][0]])


complete_Model_Prompt = f"Context from document: {context}. User prompt: {redacted_prompt}. Answer:"


response = call_model(complete_Model_Prompt)
reply = response.text
token_used = response.usage_metadata.total_token_count
model = response.model_version

print(f"{model} : {reply}", end="\n")
print(f"Token Spent: {token_used}")

gemini-3.6-flash : I cannot find that information in the provided document.
Token Spent: 811
